# 03 — Trộn tấn công với nền UNSW và tính đặc trưng cửa sổ

Theo `PLAN_tron_nen_window.md` mục 4 (bước 03).

1. **Chia task tấn công** 70/15/15 theo task, phân tầng theo kịch bản + công cụ.
   - `icmp_echo_discovery`, `tcp_ack_host_discovery`, `tcp_syn_host_discovery`: capture thiếu gói ARP quét (BPF chặn) → **chỉ test**, cờ `incomplete_capture=1`.
   - `arp_discovery` (2 task): cả hai vào train; notebook 04 đánh giá bằng leave-one-task-out.
2. **Trộn ở mức bảng gói**: mỗi task lấy một đoạn nền cùng split (đệm 300 s hai đầu), dời thời gian tấn công vào đoạn nền.
   - Persona: 80% mẫu chọn một client UNSW đang hoạt động, đổi IP → `.236`, MAC → MAC attacker.
   - Máy đích: chọn một camera UNSW đang hoạt động, đổi IP → `.182`, MAC → MAC Pi.
3. **Đặc trưng cửa sổ** theo IP nguồn, bước 10 s, 3 thang 10/60/300 s; không dùng IP/MAC/TTL/port cụ thể.
4. **Nhãn cửa sổ**: `recon`/`bruteforce` nếu host `.236` gửi ≥1 gói `attack` trong 10 s hiện tại; `post_attack` (còn tấn công trong 300 s trước) và `victim` (host `.182`) không dùng để train.
5. `sample_weight`: mỗi task tổng 1 → trong lớp, kịch bản nhận phần ∝ √(số cửa sổ) → các lớp cân bằng;
   mỗi cửa sổ không vượt `MAX_WINDOW_SHARE` tổng trọng số (tránh vài cửa sổ của kịch bản rất nhỏ chi phối loss).

Đầu ra: `Data_after_processing/windows/{train,val,test}.csv`, `reports/03_*.csv`.

In [1]:
import zlib
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = Path('/home/ubuntu/sepcung/03.Brute_Force_vs_Recon')
OUT = ROOT / 'Data_after_processing'
BEN_DIR = OUT / 'benign_packets'
WIN_DIR = OUT / 'windows'
PART_DIR = WIN_DIR / '_parts'
REPORT_DIR = OUT / 'reports'
WIN_DIR.mkdir(parents=True, exist_ok=True)
PART_DIR.mkdir(parents=True, exist_ok=True)

ATTACKER_IP, ATTACKER_MAC = '192.168.1.236', '50:bb:b5:fe:d8:b8'
TARGET_IP, TARGET_MAC = '192.168.1.182', 'e4:5f:01:a3:c6:0d'
LAN_PREFIX = '192.168.1.'

HOP = 10                  # giây — bước đánh giá trên router
SCALES = (10, 60, 300)    # giây — độ dài cửa sổ
PAD = 300                 # đệm nền trước/sau tấn công (>= thang dài nhất)
PERSONA_PROB = 0.8
MIN_DEV_PKTS = 20         # thiết bị phải gửi >= số gói này trong đoạn mới được chọn làm persona/máy đích
SEED = 42
N_JOBS = 12
CLASS_WEIGHT = {'benign': 1.0, 'recon': 1.0, 'bruteforce': 1.0}
MAX_WINDOW_SHARE = 0.001  # trần trọng số 1 cửa sổ = 0,1% tổng trọng số của split

TEST_ONLY = ['icmp_echo_discovery', 'tcp_ack_host_discovery', 'tcp_syn_host_discovery']
TRAIN_ONLY_LOTO = ['arp_discovery']

CLIENT_MACS = ['74:2f:68:81:69:42', 'ac:bc:32:d4:6f:2f', 'b4:ce:f6:a7:a3:c2', '40:f3:08:ff:1e:da',
               'd0:a6:37:df:a1:e1', 'f4:5c:89:93:cc:85', '08:21:ef:3b:fc:e3']
CAMERA_MACS = ['70:ee:50:18:34:43', 'f4:f2:6d:93:51:f1', '00:16:6c:ab:6b:88', '30:8c:fb:2f:e4:b2',
               '00:62:6e:51:27:2e', 'e8:ab:fa:19:de:4f', '30:8c:fb:b6:ea:45', '00:24:e4:11:18:a8']

task = pd.read_csv(REPORT_DIR / '01_task_inventory.csv')
days = pd.read_csv(REPORT_DIR / '02_day_summary.csv')
task = task[task['usable']].copy()
print(len(task), 'task dùng được;', len(days), 'ngày nền')

1264 task dùng được; 20 ngày nền


## 1. Chia task tấn công

In [2]:
task['strat'] = task['scenario'] + '|' + task['tool'].fillna('')
task['split'] = ''
task.loc[task['scenario'].isin(TEST_ONLY), 'split'] = 'test'
task.loc[task['scenario'].isin(TRAIN_ONLY_LOTO), 'split'] = 'train'
rest = task[task['split'] == ''].copy()
# tầng quá nhỏ (<7 task) gộp về theo kịch bản để stratify được
cnt = rest['strat'].map(rest['strat'].value_counts())
rest.loc[cnt < 7, 'strat'] = rest.loc[cnt < 7, 'scenario']
tr, tmp = train_test_split(rest.index, test_size=0.30, stratify=rest['strat'], random_state=SEED)
va, te = train_test_split(tmp, test_size=0.50, stratify=rest.loc[tmp, 'strat'], random_state=SEED)
task.loc[tr, 'split'], task.loc[va, 'split'], task.loc[te, 'split'] = 'train', 'val', 'test'
task['incomplete_capture'] = task['scenario'].isin(TEST_ONLY).astype(int)

split_tab = task.pivot_table(index='scenario', columns='split', values='task_id', aggfunc='count', fill_value=0)
split_tab.to_csv(REPORT_DIR / '03_attack_task_split.csv')
split_tab

split,test,train,val
scenario,,,
arp_discovery,0,2,0
firewall_ack_mapping,38,176,38
icmp_echo_discovery,118,0,0
ssh,44,196,42
stealth_fin_scan,2,11,3
stealth_null_scan,2,11,2
stealth_xmas_scan,2,11,2
tcp_ack_host_discovery,22,0,0
tcp_connect_scan,4,20,4


## 2. Hàm đặc trưng cửa sổ

Mỗi gói đóng hai vai: **gửi** (tính cho `ip_src`, hoặc `arp_spa` với ARP) và **nhận** (tính cho `ip_dst`/`arp_tpa`).
- Tổng trượt: cộng dồn theo khoá `(host, bucket)` rồi `searchsorted` — chính xác, O(n log n).
- Số phần tử khác nhau trượt (đích, cặp đích:port, port…): gộp các lần xuất hiện của cùng phần tử thành khoảng phủ, rồi quét sự kiện +1/−1.
- Chỉ xuất cửa sổ tại bucket host có gửi gói (router đánh giá khi host hoạt động).

In [3]:
CNT_TX = ['pkts', 'bytes', 'tcp', 'syn', 'synack', 'rst', 'fin_only', 'null', 'xmas', 'ack_only',
          'udp', 'icmp_echo', 'arp_req', 'arp_rep']
CNT_RX = ['rx_pkts', 'rx_rst', 'rx_synack', 'rx_icmp_unreach']
DISTINCT = ['d_dst', 'd_pair', 'd_port', 'd_syn_pair', 'd_arp', 'd_icmp']
SHIFT = np.int64(1) << 32


def _col(s, fill=-1):
    return pd.to_numeric(s, errors='coerce').astype('float64').fillna(fill).to_numpy().astype(np.int64)


def _str(s):
    return s.replace('', pd.NA)


def packet_table(pk):
    f = _col(pk['tcp_flags'])
    proto = _col(pk['ip_proto'])
    icmp_t = _col(pk['icmp_type'])
    arp_op = _col(pk['arp_op'], 0)
    tcp_len = _col(pk['tcp_len'], 0)
    dport = np.where(proto == 6, _col(pk['tcp_dport']), np.where(proto == 17, _col(pk['udp_dport']), -1))
    tcp, udp, icmp = proto == 6, proto == 17, proto == 1
    fv = np.where(tcp, f, -1)
    return pd.DataFrame({
        # notebook 01 lưu ô trống là '' -> coi như NA để gói ARP lấy IP từ arp_spa/arp_tpa
        'src': _str(pk['ip_src']).fillna(_str(pk['arp_spa'])).to_numpy(object),
        'dst': _str(pk['ip_dst']).fillna(_str(pk['arp_tpa'])).to_numpy(object),
        'b': (pk['ts'].to_numpy() // HOP).astype(np.int64),
        'len': _col(pk['len'], 0), 'dport': dport,
        'tcp': tcp, 'syn': tcp & ((fv & 0x12) == 0x02), 'synack': tcp & ((fv & 0x12) == 0x12),
        'rst': tcp & ((fv & 0x04) > 0), 'fin_only': tcp & ((fv & 0x17) == 0x01), 'null': tcp & (fv == 0),
        'xmas': tcp & ((fv & 0x29) == 0x29), 'ack_only': tcp & (fv == 0x10) & (tcp_len == 0),
        'udp': udp, 'icmp_echo': icmp & (icmp_t == 8), 'icmp_unreach': icmp & (icmp_t == 3),
        'arp_req': arp_op == 1, 'arp_rep': arp_op == 2,
        'attack': (pk['label'] == 'attack').to_numpy(),
    })


def _rolling_sum(keys, vals, k):
    # keys: host<<32 | b đã sắp tăng, duy nhất; tổng trên (b-k, b]
    cs = np.vstack([np.zeros((1, vals.shape[1])), np.cumsum(vals, axis=0)])
    start = np.searchsorted(keys, keys - k + 1, side='left')
    return cs[1:] - cs[start]


def _rolling_distinct(h, item, b, k, qkeys):
    if len(h) == 0:
        return np.zeros(len(qkeys))
    o = np.lexsort((b, item, h))
    h, item, b = h[o], item[o], b[o]
    new = np.r_[True, (h[1:] != h[:-1]) | (item[1:] != item[:-1]) | (b[1:] > b[:-1] + k)]
    gid = np.cumsum(new) - 1
    last_b = np.zeros(gid[-1] + 1, dtype=np.int64)
    np.maximum.at(last_b, gid, b)
    hh = h[new]
    ev_k = np.r_[hh * SHIFT + b[new], hh * SHIFT + last_b + k]
    ev_d = np.r_[np.ones(len(hh)), -np.ones(len(hh))]
    o = np.argsort(ev_k, kind='stable')
    ev_k, cs = ev_k[o], np.cumsum(ev_d[o])
    idx = np.searchsorted(ev_k, qkeys, side='right') - 1
    return np.where(idx >= 0, cs[np.maximum(idx, 0)], 0)


def window_features(pk, keep_hosts=None):
    t = packet_table(pk)
    hosts, inv = np.unique(np.r_[t['src'].fillna('').to_numpy(str), t['dst'].fillna('').to_numpy(str)], return_inverse=True)
    n = len(t)
    t['hs'], t['hd'] = inv[:n].astype(np.int64), inv[n:].astype(np.int64)
    empty = np.where(hosts == '')[0]
    if len(empty):
        t.loc[t['hs'] == empty[0], 'hs'] = -1
        t.loc[t['hd'] == empty[0], 'hd'] = -1

    tx = t[t['hs'] >= 0].assign(pkts=1, bytes=lambda d: d['len'])
    tx = tx.groupby(['hs', 'b'])[CNT_TX + ['attack']].sum().rename(columns={'attack': 'attack_tx'})
    rx = t[t['hd'] >= 0].assign(rx_pkts=1, rx_rst=lambda d: d['rst'], rx_synack=lambda d: d['synack'],
                                rx_icmp_unreach=lambda d: d['icmp_unreach'], attack_rx=lambda d: d['attack'])
    rx = rx.groupby(['hd', 'b'])[CNT_RX + ['attack_rx']].sum()
    rx.index.names = ['hs', 'b']
    base = tx.join(rx, how='outer').fillna(0).sort_index()
    h = base.index.get_level_values('hs').to_numpy(np.int64)
    bb = base.index.get_level_values('b').to_numpy(np.int64)
    keys = h * SHIFT + bb
    active = base['pkts'].to_numpy() > 0
    if keep_hosts is not None:
        keep_idx = np.where(np.isin(hosts, list(keep_hosts)))[0]
        active &= np.isin(h, keep_idx)
    qkeys = keys[active]

    out = pd.DataFrame({'host': hosts[h[active]], 'b': bb[active]})
    cnt_cols = CNT_TX + CNT_RX
    vals = base[cnt_cols].to_numpy(float)
    att = base[['attack_tx', 'attack_rx']].to_numpy(float)
    out['attack_tx_cur'] = att[active, 0]
    out['attack_trail'] = _rolling_sum(keys, att, SCALES[-1] // HOP)[active].sum(axis=1)

    txp = t[t['hs'] >= 0]
    hs_, b_ = txp['hs'].to_numpy(np.int64), txp['b'].to_numpy(np.int64)
    dst_, dport_ = txp['dst'].fillna('').to_numpy(str), txp['dport'].to_numpy()
    item_src = {
        'd_dst': (dst_ != '', dst_),
        'd_pair': (dport_ >= 0, np.char.add(np.char.add(dst_, ':'), dport_.astype(str))),
        'd_port': (dport_ >= 0, dport_.astype(str)),
        'd_syn_pair': (txp['syn'].to_numpy(), np.char.add(np.char.add(dst_, ':'), dport_.astype(str))),
        'd_arp': (txp['arp_req'].to_numpy() & (dst_ != ''), dst_),
        'd_icmp': (txp['icmp_echo'].to_numpy(), dst_),
    }
    item_codes = {}
    for name, (m, it) in item_src.items():
        codes = pd.factorize(it[m])[0].astype(np.int64)
        item_codes[name] = (hs_[m], codes, b_[m])

    for S in SCALES:
        k = S // HOP
        r = _rolling_sum(keys, vals, k)[active]
        for i, c in enumerate(cnt_cols):
            out[f'f_{c}_{S}'] = r[:, i]
        for name, (hh, it, b2) in item_codes.items():
            out[f'f_{name}_{S}'] = _rolling_distinct(hh, it, b2, k, qkeys)
        g = lambda c: out[f'f_{c}_{S}']
        out[f'f_syn_ratio_{S}'] = g('syn') / g('tcp').clip(lower=1)
        out[f'f_rx_rst_ratio_{S}'] = g('rx_rst') / g('pkts').clip(lower=1)
        out[f'f_rx_unreach_ratio_{S}'] = g('rx_icmp_unreach') / g('pkts').clip(lower=1)
        out[f'f_syn_per_pair_{S}'] = g('syn') / g('d_syn_pair').clip(lower=1)
        out[f'f_port_per_dst_{S}'] = g('d_port') / g('d_dst').clip(lower=1)
        out[f'f_bytes_per_pkt_{S}'] = g('bytes') / g('pkts').clip(lower=1)
    return out


FEATURES = [f'f_{c}_{S}' for S in SCALES for c in CNT_TX + CNT_RX + DISTINCT
            + ['syn_ratio', 'rx_rst_ratio', 'rx_unreach_ratio', 'syn_per_pair', 'port_per_dst', 'bytes_per_pkt']]
print(len(FEATURES), 'đặc trưng')

90 đặc trưng


In [4]:
# Kiểm thử nhỏ: 1 host quét 30 port trong 3 bucket -> d_port_10 tăng dần, d_port_60 cộng dồn
_toy = pd.DataFrame({
    'ts': np.r_[np.arange(10) * 1.0, 10 + np.arange(10) * 1.0, 20 + np.arange(10) * 1.0],
    'len': 60, 'eth_src': 'a', 'eth_dst': 'b', 'ip_src': '10.0.0.1', 'ip_dst': '10.0.0.2', 'ip_proto': 6, 'ttl': 64,
    'tcp_sport': 40000, 'tcp_dport': np.arange(30), 'tcp_flags': 0x02, 'tcp_len': 0, 'udp_sport': pd.NA,
    'udp_dport': pd.NA, 'icmp_type': pd.NA, 'arp_op': pd.NA, 'arp_sha': pd.NA, 'arp_spa': pd.NA, 'arp_tpa': pd.NA,
    'label': 'attack'})
_w = window_features(_toy)
assert _w['f_d_port_10'].tolist() == [10, 10, 10] and _w['f_d_port_60'].tolist() == [10, 20, 30]
assert _w['f_syn_60'].tolist() == [10, 20, 30] and _w['f_syn_per_pair_60'].tolist() == [1, 1, 1]
_w[['host', 'b', 'f_syn_10', 'f_d_port_10', 'f_d_port_60', 'f_d_dst_300', 'attack_tx_cur', 'attack_trail']]

,host,b,f_syn_10,f_d_port_10,f_d_port_60,f_d_dst_300,attack_tx_cur,attack_trail
0,10.0.0.1,0,10.0,10.0,10.0,1.0,10.0,10.0
1,10.0.0.1,1,10.0,10.0,20.0,1.0,10.0,20.0
2,10.0.0.1,2,10.0,10.0,30.0,1.0,10.0,30.0


## 3. Cửa sổ nền: toàn bộ 20 ngày (chỉ host LAN)

In [5]:
DAY_T = days.set_index('day')[['split', 't0', 't1']].to_dict('index')


def day_windows(day):
    pk = pd.read_parquet(BEN_DIR / f'{day}.parquet')
    w = window_features(pk)
    w = w[w['host'].str.startswith(LAN_PREFIX)].copy()
    w.insert(0, 'split', DAY_T[day]['split'])
    w.insert(1, 'source', 'benign_day')
    w.insert(2, 'day', day)
    w['sample_id'] = 'day_' + day
    w['label'] = 'benign'
    w.to_parquet(PART_DIR / f'day_{day}.parquet', index=False)
    return day, len(w)


with ProcessPoolExecutor(max_workers=10) as ex:
    day_res = list(ex.map(day_windows, days['day']))
pd.DataFrame(day_res, columns=['day', 'windows']).set_index('day').T

day,16-09-23,16-09-24,16-09-25,16-09-26,16-09-27,16-09-28,16-09-29,16-09-30,16-10-01,16-10-02,16-10-03,16-10-04,16-10-05,16-10-06,16-10-07,16-10-08,16-10-09,16-10-10,16-10-11,16-10-12
windows,72171,68505,68835,68515,68200,82004,88202,88344,87719,83947,87883,85867,89691,79499,79395,73091,70458,72365,75708,43646


## 4. Mẫu trộn: mỗi task tấn công + một đoạn nền cùng split

In [6]:
def _remap(df, ip_old, mac_old, ip_new, mac_new):
    for c in ['ip_src', 'ip_dst', 'arp_spa', 'arp_tpa']:
        df[c] = df[c].mask(df[c] == ip_old, ip_new)
    for c in ['eth_src', 'eth_dst', 'arp_sha']:
        df[c] = df[c].mask(df[c] == mac_old, mac_new)
    return df


def _pick(ben, macs, rng):
    lan = ben[ben['eth_src'].isin(macs) & ben['ip_src'].fillna('').str.startswith(LAN_PREFIX)]
    vc = lan['eth_src'].value_counts()
    vc = vc[vc >= MIN_DEV_PKTS]
    if vc.empty:
        return None, None
    mac = rng.choice(vc.index.to_numpy())
    ip = lan.loc[lan['eth_src'] == mac, 'ip_src'].mode().iat[0]
    return mac, ip


def build_sample(job):
    rng = np.random.default_rng(job['seed'])
    att = pd.read_parquet(job['parquet'])
    att = att[att['label'].isin(['attack', 'aux'])].copy()
    a0, a1 = job['attack_t0'], job['attack_t1']
    seg_len = (a1 - a0) + 2 * PAD
    d0, d1 = DAY_T[job['day']]['t0'], DAY_T[job['day']]['t1']
    seg0 = rng.uniform(d0, max(d0, d1 - seg_len))
    att['ts'] = att['ts'] + (seg0 + PAD - a0)
    ben = pd.read_parquet(BEN_DIR / f"{job['day']}.parquet",
                          filters=[('ts', '>=', seg0), ('ts', '<', seg0 + seg_len)])
    persona_mac = persona_ip = None
    if rng.random() < PERSONA_PROB:
        persona_mac, persona_ip = _pick(ben, CLIENT_MACS, rng)
    target_mac, target_ip = _pick(ben, CAMERA_MACS, rng)
    if persona_mac:
        ben = _remap(ben, persona_ip, persona_mac, ATTACKER_IP, ATTACKER_MAC)
    if target_mac:
        ben = _remap(ben, target_ip, target_mac, TARGET_IP, TARGET_MAC)
    pk = pd.concat([ben, att[ben.columns]], ignore_index=True).sort_values('ts', kind='stable')
    w = window_features(pk, keep_hosts=[ATTACKER_IP, TARGET_IP])
    is_att = w['host'] == ATTACKER_IP
    w['label'] = np.select(
        [is_att & (w['attack_tx_cur'] > 0), is_att & (w['attack_trail'] > 0), ~is_att & (w['attack_trail'] > 0)],
        [job['class'], 'post_attack', 'victim'], default='benign')
    w.insert(0, 'split', job['split'])
    w.insert(1, 'source', 'mix')
    w.insert(2, 'day', job['day'])
    w['sample_id'] = job['sample_id']
    w['scenario'], w['task_id'] = job['scenario'], job['task_id']
    w['incomplete_capture'] = job['incomplete_capture']
    w['persona'] = persona_mac or ''
    w['target_dev'] = target_mac or ''
    w.to_parquet(PART_DIR / f"{job['sample_id']}.parquet", index=False)
    return {'sample_id': job['sample_id'], 'scenario': job['scenario'], 'task_id': job['task_id'],
            'split': job['split'], 'day': job['day'], 'persona': persona_mac or '', 'target_dev': target_mac or '',
            'bg_pkts': len(ben), 'attack_pkts': int((att['label'] == 'attack').sum()),
            **{f'win_{l}': int((w['label'] == l).sum()) for l in ['benign', job['class'], 'post_attack', 'victim']}}


split_days = days.groupby('split')['day'].apply(list).to_dict()
jobs = []
for r in task.to_dict('records'):
    seed = zlib.crc32(f"{r['scenario']}/{r['task_id']}/{SEED}".encode())
    rng = np.random.default_rng(seed)
    jobs.append({'sample_id': f"{r['scenario']}_{int(r['task_id']):04d}", 'scenario': r['scenario'],
                 'task_id': r['task_id'], 'class': r['class'], 'split': r['split'],
                 'day': rng.choice(split_days[r['split']]), 'seed': seed, 'parquet': r['parquet'],
                 'attack_t0': r['attack_t0'], 'attack_t1': r['attack_t1'],
                 'incomplete_capture': r['incomplete_capture']})
jobs.sort(key=lambda j: -(j['attack_t1'] - j['attack_t0']))  # task dài chạy trước
with ProcessPoolExecutor(max_workers=N_JOBS) as ex:
    mix_res = pd.DataFrame(list(ex.map(build_sample, jobs)))
mix_res.to_csv(REPORT_DIR / '03_mix_samples.csv', index=False)
mix_res.groupby(['scenario', 'split'])[['bg_pkts', 'attack_pkts'] + [c for c in mix_res if c.startswith('win_')]].sum()

/tmp/ipykernel_2698071/2725517438.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['scenario'], w['task_id'] = job['scenario'], job['task_id']


/tmp/ipykernel_2698071/2725517438.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['incomplete_capture'] = job['incomplete_capture']


/tmp/ipykernel_2698071/2725517438.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['persona'] = persona_mac or ''


/tmp/ipykernel_2698071/2725517438.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['target_dev'] = target_mac or ''


/tmp/ipykernel_2698071/2725517438.py:49: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['scenario'], w['task_id'] = job['scenario'], job['task_id']


/tmp/ipykernel_2698071/2725517438.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['incomplete_capture'] = job['incomplete_capture']


... (5,047 outputs left out)

/tmp/ipykernel_2698071/2725517438.py:50: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['incomplete_capture'] = job['incomplete_capture']


/tmp/ipykernel_2698071/2725517438.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['persona'] = persona_mac or ''


/tmp/ipykernel_2698071/2725517438.py:52: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  w['target_dev'] = target_mac or ''


bg_pkts  attack_pkts  win_benign  \
scenario               split                                     
arp_discovery          train     7862          970          79   
firewall_ack_mapping   test    245800        27587        1034   
                       train  2712003        97308        4430   
                       val     157438        10882        1033   
icmp_echo_discovery    test   1250121         1363        3787   
ssh                    test    413320     14213760        1236   
                       train  1574405     33899666        5353   
                       val     478142     13313956        1208   
stealth_fin_scan       test     10311         6036          74   
                       train    48531        24220         317   
                       val      10767         3102          86   
stealth_null_scan      test      7298         6036          61   
                       train    90487        27184         276   
                       val      14359         6023          74   
stealth_xmas_scan      test      8242           45          54   
                       train    84324        24149         297   
                       val       7844         8990          40   
tcp_ack_host_discovery test    137431          630         699   
tcp_connect_scan       test     16068        11815          79   
                       train   148378        60560         530   
                       val      19830        13985         109   
tcp_syn_host_discovery test     35222           81         148   
tcp_syn_scan           test    500349       910608        1595   
                       train  5093124      6691236        6790   
                       val     349758      1560787        1596   
udp_port_scan          test    101530         4710         489   
                       train  1416616       195239        2129   
                       val      82356        54432         453   

                              win_bruteforce  win_post_attack  win_victim  \
scenario               split                                                
arp_discovery          train             0.0               13          61   
firewall_ack_mapping   test              0.0              196         875   
                       train             0.0              701        3947   
                       val               0.0              142         897   
icmp_echo_discovery    test              0.0              773        2719   
ssh                    test           1383.0              282        2347   
                       train          6300.0              831       10636   
                       val            1280.0              166        2234   
stealth_fin_scan       test              0.0               14          80   
                       train             0.0               76         297   
                       val               0.0                8          79   
stealth_null_scan      test              0.0                0          67   
                       train             0.0               62         239   
                       val               0.0               14          67   
stealth_xmas_scan      test              0.0                0          54   
                       train             0.0               40         294   
                       val               0.0                0          52   
tcp_ack_host_discovery test              0.0              172         537   
tcp_connect_scan       test              0.0                8          83   
                       train             0.0              121         441   
                       val               0.0               12          94   
tcp_syn_host_discovery test              0.0               19         173   
tcp_syn_scan           test              0.0              358        1842   
                       train             0.0              834        9289   
                       val               0.0   

## 5. Gộp, gán `use_train` và `sample_weight`, xuất CSV

In [7]:
META = ['split', 'source', 'day', 'sample_id', 'scenario', 'task_id', 'incomplete_capture', 'persona', 'target_dev',
        'host', 'b', 'label', 'use_train', 'sample_weight', 'attack_tx_cur', 'attack_trail']
parts = sorted(PART_DIR.glob('*.parquet'))
allw = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
for c, v in [('scenario', ''), ('task_id', -1), ('incomplete_capture', 0), ('persona', ''), ('target_dev', '')]:
    allw[c] = allw[c].fillna(v) if c in allw else v
allw['use_train'] = allw['label'].isin(['benign', 'recon', 'bruteforce']).astype(int)


def weights(df):
    w = pd.Series(0.0, index=df.index)
    u = df['use_train'] == 1
    att = u & (df['label'] != 'benign')
    # mỗi mẫu (task) tổng 1 -> kịch bản tổng ∝ sqrt(số cửa sổ) trong lớp -> mỗi lớp tổng CLASS_WEIGHT
    scen = df.loc[att, 'scenario']
    w[att] = 1.0 / df[att].groupby('sample_id')['label'].transform('size')
    w[att] = w[att] / w[att].groupby(scen).transform('sum') * np.sqrt(scen.map(scen.value_counts()))
    ben = u & (df['label'] == 'benign')
    w[ben] = 1.0
    masks = {cls: u & (df['label'] == cls) for cls in CLASS_WEIGHT}

    def class_norm(w):
        for cls, m in masks.items():
            if m.any():
                w[m] *= CLASS_WEIGHT[cls] / w[m].sum()
        return w

    w = class_norm(w)
    for _ in range(5):  # trần mỗi cửa sổ rồi cân lại lớp
        w = class_norm(w.clip(upper=MAX_WINDOW_SHARE * w.sum()))
    return w * (u.sum() / w.sum())


allw['sample_weight'] = 0.0
for s in ['train', 'val']:
    m = allw['split'] == s
    allw.loc[m, 'sample_weight'] = weights(allw[m])

summary = (allw.groupby(['split', 'source', 'label'])
           .agg(windows=('b', 'size'), weight_sum=('sample_weight', 'sum')).reset_index())
summary.to_csv(REPORT_DIR / '03_window_label_summary.csv', index=False)
summary

,split,source,label,windows,weight_sum
0,test,benign_day,benign,319804,0.000000
1,test,mix,benign,9256,0.000000
2,test,mix,bruteforce,1383,0.000000
3,test,mix,post_attack,1937,0.000000
4,test,mix,recon,1171,0.000000
5,test,mix,victim,9149,0.000000
6,train,benign_day,benign,906247,305470.149426
7,train,mix,benign,20201,6809.183907
8,train,mix,bruteforce,6300,312279.333333
9,train,mix,post_attack,3028,0.000000


In [8]:
for s in ['train', 'val', 'test']:
    allw.loc[allw['split'] == s, META + FEATURES].to_csv(WIN_DIR / f'{s}.csv', index=False)
for p in parts:
    p.unlink()
PART_DIR.rmdir()
{s: (WIN_DIR / f'{s}.csv').stat().st_size // 2**20 for s in ['train', 'val', 'test']}  # MB

{'train': 493, 'val': 162, 'test': 171}

In [9]:
# Kiểm tra nhanh độ tách: median một số đặc trưng theo nhãn (train)
chk_cols = ['f_pkts_10', 'f_d_port_60', 'f_d_dst_300', 'f_syn_per_pair_60', 'f_rx_rst_ratio_60', 'f_d_arp_60', 'f_d_icmp_300']
med = allw[allw['split'] == 'train'].groupby(['label', 'scenario'])[chk_cols].median().round(2)
med.to_csv(REPORT_DIR / '03_feature_median_by_label.csv')
med

f_pkts_10  f_d_port_60  f_d_dst_300  \
label       scenario                                                    
benign                                  3.0          1.0          3.0   
            arp_discovery               2.0          1.0          2.0   
            firewall_ack_mapping        3.0          1.0          3.0   
            ssh                         3.0          1.0          3.0   
            stealth_fin_scan            3.0          1.0          2.0   
            stealth_null_scan           4.0          2.0          5.0   
            stealth_xmas_scan           4.0          1.0          3.0   
            tcp_connect_scan            2.0          1.0          3.0   
            tcp_syn_scan                3.0          1.0          3.0   
            udp_port_scan               3.0          1.0          3.0   
bruteforce  ssh                        83.0          1.0          2.0   
post_attack arp_discovery               4.0          2.0        260.0   
            firewall_ack_mapping        4.0          3.0         13.0   
            ssh                         4.0          3.0         12.0   
            stealth_fin_scan            2.0          2.0          8.0   
            stealth_null_scan           4.5          4.0         12.0   
            stealth_xmas_scan           3.0          3.0         17.0   
            tcp_connect_scan            4.0          3.0         17.0   
            tcp_syn_scan                3.0          3.0         11.0   
            udp_port_scan               4.0          3.0         11.0   
recon       arp_discovery             455.0          2.0        252.0   
            firewall_ack_mapping       12.5          2.5          2.0   
            stealth_fin_scan          102.0        290.0          2.0   
            stealth_null_scan         190.0        460.0          1.0   
            stealth_xmas_scan         140.0        425.0          1.0   
            tcp_connect_scan          950.0        958.0          2.0   
            tcp_syn_scan              112.0        232.0          2.0   
            udp_port_scan              26.0          7.0          2.0   
victim      arp_discovery               2.0          1.0          4.0   
            firewall_ack_mapping        3.0          1.0          4.0   
            ssh                        26.0          9.0          4.0   
            stealth_fin_scan            7.0          1.0          4.0   
            stealth_null_scan           7.0          3.0          6.0   
            stealth_xmas_scan           8.0          2.0          4.0   
            tcp_connect_scan            2.0          2.0          4.0   
            tcp_syn_scan               10.0          3.0          4.0   
            udp_port_scan               3.0          1.0          4.0   

                                  f_syn_per_pair_60  f_rx_rst_ratio_60  \
label       scenario                                                     
benign                                         0.00               0.00   
            arp_discovery                      0.00               0.00   
            firewall_ack_mapping               0.00               0.00   
            ssh                                0.00               0.00   
            stealth_fin_scan                   0.00               0.00   
            stealth_null_scan                  0.00               0.00   
            stealth_xmas_scan                  0.00               0.00   
            tcp_connect_scan                   0.00               0.00   
            tcp_syn_scan                       0.00               0.00   
            udp_port_scan                      0.00               0.00   
bruteforce  ssh                               20.00               0.14   
post_attack arp_discovery                      1.00               0.05   
            firewall_ack_mapping               1.00               0.00   
            ssh                                1.00               0.00   
  